# Prospective native joint DiT semantic screen
Fresh native task-one-to-task-two training with linear activation, half the original LR, classifier KD 4 and semantic CE 0. **Old-row CE remains native weight 1**. The diagnostic weight5 adapter is qualified only for a restored task-two fork, so this distinct candidate cannot be assigned the 63.3% result.

DESIGN displays the complete configuration without training. LIVE uses the existing native capacity runner and prearmed stopping observer: one seed 101 stream, stopped after authenticated committed task two. The full five-task schedule and split are preserved. Preparation writes 12 metadata cells; only the learned seed 101 cell is launched. Three seeds are reserved for future finalists.

In [1]:
from pathlib import Path
import json
import sys


ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "files/notebooks/thesis/candidate_20261014.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from files.notebooks.thesis.candidate_20261014 import read_asset, remote_only


remote_only()
architecture = read_asset("architecture.json")
evidence = read_asset("evidence.json")
screen = read_asset("screen_result.json")


In [2]:
MODE = "DESIGN"  # DESIGN or LIVE
RUN_NAME = "thesis_native_linear_kd4_ce0_oldce1_20261014_v1"
SOURCE_PLAN = Path("/dev/shm/dit_local_storage/new_h100_a_native_classifier_axis_v1/artifacts/group/plan.json")
SOURCE_PLAN_SHA256 = "0f6c458c9f79d887f1715fc6f72dd36f4db04a9ab611557fc19790ec0cde5388"
print("Full configuration:", ROOT / "files/notebooks/thesis/candidate_20261014/prospective_native.yaml")
print("Prospective result unavailable; native old-row CE weight1.")


Full configuration: /tmp/thesis_notebook_validation_20261014/files/notebooks/thesis/candidate_20261014/prospective_native.yaml
Prospective result unavailable; native old-row CE weight1.


## Computational architecture
Input is CIFAR-10 RGB, 32 x 32. The wrapper owns standardization: raw real and generated images enter its preprocessing contract once. The CNN patchifier uses patch size 2, producing a 16 x 16 token grid at width 128 with additive fixed 2D sine/cosine positions. Time and class embeddings condition the implemented adaptive-normalization blocks. Diffusion horizon: 1,000.

The denoising backbone has **8 blocks, 4 attention heads, FFN expansion 4**. The classifier branch has **4 blocks, 4 heads, FFN expansion 4**. Feature aggregation is {"1": [-1]}; classifier connections are {"-1": [-1]}. These are native layer-ID maps, not additional external networks. Forced global average pooling overrides token-only readout flags. The classifier hidden MLP has ratio 1, **linear activation and dropout 0.5**. Its probabilities remain softmax outputs: linear changes the hidden activation, not normalization. Drop-path is zero; the refinement CNN is disabled.

The continual wrapper maintains seen-class mapping and expands outputs natively. The same persistent **raw joint DiTClassifier** performs classification and epsilon prediction for replay. EMA is disabled. A separate U-Net is an auxiliary noise teacher only; it does not generate replay instead of the joint network. Full generic options, source-level constructor defaults and implementation digests are stored in architecture.json.

## Training and inference
Fixed five-task schedule: [7,3] -> [2,8] -> [5,6] -> [9,4] -> [0,1], data seed 42. Screening seed is 101; three model seeds are reserved for finalists. A stratified 10% TRAIN holdout supplies DEV. Classification uses all seen classes without task ID. The reported diagnostic evaluates 2,000 DEV examples (500/class), not official test.

Joint training: Adam, constant LR **0.0009430051881577431**, batch **128**, clipnorm **5**. Coefficients: primary CE **0.00215**, epsilon MSE **1**, previous-classifier soft KD **0.0043** (temperature 2, old-class scope), noise KD **0.5**. Previous/current noise-teacher weights are 1/1; classifier-teacher weights are 1/0. Student CE and classifier KD use noisy input capped at timestep 64, null-only class conditioning and all rows. Null/timestep masking is off; denoising uses the full 1,000-step range. Task one has no previous-class teacher.

The best diagnostic additionally weights old rows **5 times inside primary training CE**, with per-batch normalization by the sum of weights. KD, denoising and inference are unchanged. This is a qualified diagnostic adapter, not a full-history YAML option; it balances aggregate CE mass for the 500-old/2,500-current pool.

Native prospective budgets are 100 epochs/task. Each task first trains a fresh current-class U-Net noise specialist on 9,000 permitted TRAIN rows, then the joint student on 2,500 current rows plus 500 generated old rows after task one. U-Net widths: 64/128/256; block depth 3; bottleneck width 384 / depth 2; GELU, batch normalization, skip connections, average-pool downsampling and transposed-convolution upsampling. Teacher AdamW LR **0.0009232173954634668**, weight decay **0.000010569549424860102**, batch 128. These native budgets differ from the best diagnostic, which restores task one and applies **240 additional joint updates** before semantic phases.

Replay uses the raw joint network: **50 DDIM steps, eta 0, CFG 2.929069189794846**, uniform selection, fixed total 500 old images. Inference uses clean timestep 0, primary-head weight 1, distillation-head weight 0 and no timestep ensemble. Sampling/postprocessing and later classifier preprocessing follow the wrapper contract once.

## Semantic procedure
This uses existing **RouteController/RoutePhase**, a TMCL-inspired adaptation, not the separate native TMCLTrainer VI/MI protocol. Acquisition freezes the main network while learning task/class modulators with contrastive and orthogonality objectives. Consolidation freezes the acquired target and modulators, updating permitted semantic weights through representation alignment.

The candidate uses **100 acquisition + 100 consolidation updates**, separate Adam LR **0.001**, batch **32**, **4 TMCL-style augmentation views**, temperature **0.1**, alignment **1**, orthogonality **1**, and consolidation CE **0**. Phase noise levels are zero. Modulators are retained during training; ordinary inference is unmodulated. The reported late fork builds a fresh four-class bank; it does not reconstruct two-task gate history.

Only extra consolidation CE is removed. Ordinary joint classifier CE remains active. This computational design is not evidence of a biological mechanism or an exact JDCL/TMCL reproduction.

## Differences and capacity
In this experiment old CE is 1, semantic phases start at task one, the linear classifier is initialized from scratch, and the student receives 100 complete epochs/task. The late screen instead changes the restored classifier function and trains 240 updates. Cross-protocol differences are therefore descriptive.

Native task one/task two have 2,000/2,400 joint updates, 7,100 specialist updates each, and 100 + 100 semantic updates each. The screen reports clean all-seen-class DEV, not official test.

One worker reserves 32 GiB TensorFlow, 24 GiB host RAM including scratch, and 6 CPU cores with 2/1/2 thread settings. Native admission checks actual UUID, runtime, ownership and memory. The original H100 host/cache are required; replacement hosts need fresh qualification. A reservation is not proof of currently free capacity.

In [3]:
from files.notebooks.thesis.candidate_20261014 import prepare_native_history, execute_prepared


if MODE == "LIVE":
    receipt = prepare_native_history(SOURCE_PLAN, SOURCE_PLAN_SHA256, RUN_NAME)
    print("Prepared:", receipt["plan"], receipt["plan_sha256"])
    execute_prepared(receipt)
    print("Authenticated task-two status:", receipt["results"])
    print("Intentionally incomplete as a five-task run.")
elif MODE == "DESIGN":
    print("Design displayed; no new stream prepared, trained, sampled or scored.")
else:
    raise ValueError("Choose DESIGN or LIVE explicitly.")


Design displayed; no new stream prepared, trained, sampled or scored.


## Reporting
Retain the per-task accuracy matrix, task one accuracy, old/new accuracy at task two, forgetting in percentage points, counts, phase-update ledgers, all failures and source/config/checkpoint identities. Only after genuine five-task completion compute final average accuracy, incremental accuracy, forgetting and local backward transfer within each seed, then summarize three finalist seeds.

Historic official-test HPO exposure remains disclosed. The DEV record cannot become an independent test result by relabelling it. The final evaluation protocol and its actual observations still need to be registered and executed.